# Imports

In [4]:
from transformers import AutoTokenizer, BertForMultipleChoice
import pandas as pd
#from transformers import pipeline
from tqdm import tqdm
from datasets import load_dataset, load_from_disk

# restrict trainer to not use all cores
import os
#os.environ["OPENMP_NUM_THREADS"] = "3"

# Load Dataset
If running on google colab, import data from google drive folder titled BERT, otherwise load from data directory. Loads the pre-labeled dataset produced by BERT-data-process notebook

In [ ]:
try:
    from google.colab import drive
    data_path = "./bert-ds-labeled"
    model_path = "./models/"
    colab = True
except
    data_path = "../data/bert-ds-labeled"
    model_path = "../models/"
    colab = False

if colab:
    drive.mount('/content/drive')
    %cd drive/MyDrive/BERT
    !pwd

In [5]:
dataset = load_from_disk(data_path)
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 160028
    })
    validation: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1000
    })
})

In [6]:
dataset['train'][0:4]

{'labels': [3, 2, 3, 2],
 'inputs': ['CallOfDuty. ... This is chilling',
  'Google. one',
  'ApexLegends. literally toxic bro, came out of the game when I was clear to him. Oh and I hit the same person three times to talk about lmao disappointment.',
  'WorldOfCraft. Take a look at this article I just got! [Uncanny Combat Gloves of the Incomparable Fighter]']}

In [ ]:
sentiments = ["Irrelevant","Positive","Neutral","Negative"]

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 160028
    })
    validation: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1000
    })
})

In [11]:
dataset["train"][0:4]

{'labels': [3, 2, 3, 2],
 'inputs': ['CallOfDuty. ... This is chilling',
  'Google. one',
  'ApexLegends. literally toxic bro, came out of the game when I was clear to him. Oh and I hit the same person three times to talk about lmao disappointment.',
  'WorldOfCraft. Take a look at this article I just got! [Uncanny Combat Gloves of the Incomparable Fighter]']}

### Tokenize

In [12]:
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
# adapted from https://huggingface.co/docs/transformers/tasks/multiple_choice

count = 0
total = int(len(dataset["train"])/1000)

def preprocess(examples):
    global count, total
    count += 1
    tweet_text = [[content] * 4 for content in examples["inputs"]]
    #entities = examples["Entity"]

    tweet_sentiment = [
        [*sentiments] for label in examples["labels"]
    ]

    tweet_text = sum(tweet_text, [])
    tweet_sentiment = sum(tweet_sentiment, [])

    print(count,"/",total,"\t", len(tweet_text), len(tweet_sentiment))

    tokenized_examples = tokenizer(tweet_text, tweet_sentiment, truncation = True)
    return {k: [v[i : i+4] for i in range(0,len(v),4)] for k, v in tokenized_examples.items()}

In [14]:
tokenized_data = dataset.map(preprocess, batched = True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

1 / 160 	 4000 4000


In [15]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['labels', 'inputs', 'input_ids', 'attention_mask'],
        num_rows: 160028
    })
    validation: Dataset({
        features: ['labels', 'inputs', 'input_ids', 'attention_mask'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['labels', 'inputs', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
})

# Training
### Construct Model

In [16]:
from transformers import DataCollatorForMultipleChoice
collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

In [ ]:
if colab:
    !pip install evaluate

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [19]:
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer

model = AutoModelForMultipleChoice.from_pretrained('distilbert/distilbert-base-uncased')#"google-bert/bert-base-uncased")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForMultipleChoice were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

### Train

In [22]:
run_name = "lr2Eminus6"
os.mkdir(model_path + run_name)

training_args = TrainingArguments(
    output_dir=model_path + run_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer Initialized")

output = trainer.train()
output

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Trainer Initialized


Epoch,Training Loss,Validation Loss,Accuracy
1,0.876700,0.805823,0.694667
2,0.682000,0.645796,0.759333
3,0.542600,0.512632,0.814000
4,0.419200,0.435329,0.845333
5,0.336600,0.370833,0.868000
6,0.283500,0.336668,0.882667
7,0.249300,0.318494,0.897333
8,0.220100,0.305934,0.898000
9,0.211300,0.302102,0.910000
10,0.191000,0.303720,0.908000


TrainOutput(global_step=100020, training_loss=0.44177341643773754, metrics={'train_runtime': 19339.6515, 'train_samples_per_second': 82.746, 'train_steps_per_second': 5.172, 'total_flos': 1.4025808683785933e+17, 'train_loss': 0.44177341643773754, 'epoch': 10.0})

In [23]:
model.save_pretrained("drive/MyDrive/BERT/models/basic/lr2neg6")